# 02 — Data Quality Analysis

**Goal:** Turn the red flags spotted in `01_dataset_exploration.ipynb` into concrete, countable data-quality checks, so we know exactly how many records violate each rule before designing the Silver-layer validation logic.

Each check below maps directly to a rule in `docs/silver-layer-specification.md`:

| Check | Spec Rule |
|---|---|
| Duplicate records | — |
| Negative fare amount | SLV-006 |
| Negative trip distance | SLV-004 |
| Invalid passenger count (≤ 0) | SLV-005 |
| Dropoff before pickup | SLV-003 |
| Trip distance outliers | SLV-010 |

## Step 1 — Reload the dataset

Same raw Parquet file as notebook 01, loaded fresh so this notebook can run independently.

In [2]:
import os
import pandas as pd
# Priliminary check to see if the file exists in the current working directory
# Print where Python currently thinks it is
print("Current Working Directory:", os.getcwd())

# Check if the file actually exists on disk
file_path = r"..\data\\raw\\yellow_taxi\\2024-01.parquet"
print("File exists?:", os.path.exists(file_path))


df = pd.read_parquet(file_path, engine="pyarrow")

Current Working Directory: c:\Additional_data\rbs\openscale\notebooks
File exists?: True


## Step 2 — Duplicate record check

Exact-duplicate rows would inflate trip counts and skew every downstream aggregate (revenue, demand, etc.). We check with `df.duplicated().sum()` before deciding whether de-duplication needs to be part of the Silver layer.

In [3]:
df.duplicated().sum()

np.int64(0)

## Step 3 — Negative fare amounts (SLV-006)

Counts rows where `fare_amount < 0`. Per the spec, these aren't auto-rejected — they may be legitimate refunds, disputes, or billing corrections — so they're flagged as `WARNING` and routed to quarantine rather than dropped.

In [4]:
(df["fare_amount"] < 0).sum()

np.int64(37448)

## Step 4 — Negative trip distances (SLV-004)

A negative distance is physically impossible and is a `CRITICAL` rule in the spec — any such record should be rejected outright, not quarantined.

In [5]:
(df["trip_distance"] < 0).sum()

np.int64(0)

## Step 5 — Invalid passenger counts (SLV-005)

Counts trips with `passenger_count <= 0`. The spec treats this as critical but quarantines rather than hard-rejects, since it may reflect a sensor/reporting issue rather than a fabricated trip.

In [6]:
(df["passenger_count"] <= 0).sum()

np.int64(31465)

## Step 6 — Dropoff-before-pickup timestamps (SLV-003)

A trip can't end before it starts. We count rows where `tpep_dropoff_datetime < tpep_pickup_datetime` — these are unambiguous, critical data errors and should be rejected.

In [7]:
(
    df["tpep_dropoff_datetime"]
    <
    df["tpep_pickup_datetime"]
).sum()

np.int64(56)

## Step 7 — Trip distance outlier thresholds (SLV-010)

Rather than eyeballing the max (312,722 miles), we look at the 95th/99th/99.9th percentiles to set a defensible, data-driven cutoff. The result (95th ≈ 13.7mi, 99th = 20mi, 99.9th ≈ 29.5mi) is what justified the spec's conservative 100-mile quarantine threshold.

In [ ]:
df["trip_distance"].quantile(
    [0.95, 0.99, 0.999]
)

0.950    13.69000
0.990    20.00000
0.999    29.50377
Name: trip_distance, dtype: float64

: 

## Summary & Next Steps

**Findings:**

| Check | Count | Spec Rule | Action |
|---|---:|---|---|
| Duplicate rows | 0 | — | — |
| Negative fare amount | 37,448 | SLV-006 | Quarantine |
| Negative trip distance | 0 | SLV-004 | Reject |
| Passenger count ≤ 0 | 31,465 | SLV-005 | Quarantine |
| Dropoff before pickup | 56 | SLV-003 | Reject |
| Distance > ~30mi (99.9th pct) | outlier tail | SLV-010 | Quarantine (>100mi) |

No exact duplicates and no negative distances — good news. The negative-fare and low-passenger-count counts are non-trivial (~1.3% and ~1.1% of rows respectively) and confirm the quarantine-rather-than-reject approach in the spec.

**Next:** `03_silver_layer_prototype.ipynb` implements these rules against the dataset and produces the clean (`data/silver/trips/`) and quarantined (`data/silver/quarantine/`) outputs, mirroring `spark/silver/validator.py`.